# ML Metadata Walkthrough: Wine Quality Dataset

This notebook demonstrates how to use [ML Metadata (MLMD)](https://www.tensorflow.org/tfx/guide/mlmd) to track the lineage of artifacts, executions, and events in a machine learning pipeline — independent of TFX.

**Key modifications from the original lab:**
- **Different dataset**: [UCI Wine Quality (Red Wine)](https://archive.ics.uci.edu/ml/datasets/wine+quality) instead of Chicago Taxi
- **SQLite persistent store** instead of a fake (in-memory) database
- **Statistics tracked as a separate artifact** with its own execution type
- **Anomaly detection step**: compares train vs. eval statistics and records anomalies as an artifact
- **Richer context**: tracks pipeline version, dataset origin, and experiment notes

The Wine Quality dataset contains physicochemical properties of red wine variants along with a quality rating (0–10). We will use TFDV to generate statistics, infer a schema, detect anomalies between train and eval splits, and record every step in the MLMD store.

## Imports

In [ ]:
import os
import pandas as pd

from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2

import tensorflow as tf
print('TF version:', tf.__version__)

import tensorflow_data_validation as tfdv
print('TFDV version:', tfdv.version.__version__)

## Explore the Dataset

The Wine Quality dataset has already been split into `data/train/data.csv` and `data/eval/data.csv`. Let's preview it.

In [ ]:
train_df = pd.read_csv('./data/train/data.csv')
eval_df = pd.read_csv('./data/eval/data.csv')

print(f'Train shape: {train_df.shape}')
print(f'Eval shape:  {eval_df.shape}')
print(f'\nFeatures: {list(train_df.columns)}')
train_df.head()

In [ ]:
train_df.describe()

## Process Outline

ML Metadata uses the following data model:

| Component | Description |
|---|---|
| **ArtifactType** | Describes an artifact's type and its properties |
| **Artifact** | A specific instance of an ArtifactType (e.g., a dataset, schema) |
| **ExecutionType** | Describes a component/step type and its runtime parameters |
| **Execution** | A record of a component run or step |
| **Event** | Links artifacts to executions (input/output relationships) |
| **ContextType** | Describes a conceptual group of artifacts and executions |
| **Context** | An instance of a ContextType (e.g., an experiment) |
| **Attribution** | Links artifacts to contexts |
| **Association** | Links executions to contexts |

Our pipeline will track these steps:

1. Define the MLMD storage database (SQLite)
2. Register artifact types (DataSet, Statistics, Schema, Anomalies)
3. Register execution types (StatisticsGen, SchemaGen, AnomalyDetection)
4. Run StatisticsGen → generate & record train statistics
5. Run SchemaGen → infer & record schema from statistics
6. Run AnomalyDetection → compare eval data against schema
7. Set up context, attributions, and associations
8. Query the metadata store to trace lineage

## Step 1: Define ML Metadata's Storage Database (SQLite)

Unlike the original lab which uses a fake (in-memory) database, we use **SQLite** so that metadata persists across sessions.

In [ ]:
DB_PATH = './metadata/mlmd.sqlite'
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = DB_PATH
connection_config.sqlite.connection_mode = 3  # READWRITE_OPENCREATE

store = metadata_store.MetadataStore(connection_config)
print(f'MetadataStore initialized with SQLite at: {DB_PATH}')

## Step 2: Register Artifact Types

We register four artifact types: **DataSet**, **Statistics**, **Schema**, and **Anomalies**.

In [ ]:
# --- DataSet artifact type ---
data_artifact_type = metadata_store_pb2.ArtifactType()
data_artifact_type.name = 'DataSet'
data_artifact_type.properties['name'] = metadata_store_pb2.STRING
data_artifact_type.properties['split'] = metadata_store_pb2.STRING
data_artifact_type.properties['version'] = metadata_store_pb2.INT
data_artifact_type.properties['num_rows'] = metadata_store_pb2.INT
data_artifact_type_id = store.put_artifact_type(data_artifact_type)

# --- Statistics artifact type ---
stats_artifact_type = metadata_store_pb2.ArtifactType()
stats_artifact_type.name = 'Statistics'
stats_artifact_type.properties['name'] = metadata_store_pb2.STRING
stats_artifact_type.properties['split'] = metadata_store_pb2.STRING
stats_artifact_type.properties['version'] = metadata_store_pb2.INT
stats_artifact_type_id = store.put_artifact_type(stats_artifact_type)

# --- Schema artifact type ---
schema_artifact_type = metadata_store_pb2.ArtifactType()
schema_artifact_type.name = 'Schema'
schema_artifact_type.properties['name'] = metadata_store_pb2.STRING
schema_artifact_type.properties['version'] = metadata_store_pb2.INT
schema_artifact_type_id = store.put_artifact_type(schema_artifact_type)

# --- Anomalies artifact type ---
anomalies_artifact_type = metadata_store_pb2.ArtifactType()
anomalies_artifact_type.name = 'Anomalies'
anomalies_artifact_type.properties['name'] = metadata_store_pb2.STRING
anomalies_artifact_type.properties['split'] = metadata_store_pb2.STRING
anomalies_artifact_type.properties['has_anomalies'] = metadata_store_pb2.STRING
anomalies_artifact_type_id = store.put_artifact_type(anomalies_artifact_type)

print('Registered Artifact Types:')
for at in store.get_artifact_types():
    print(f'  - {at.name} (ID: {at.id})')

## Step 3: Register Execution Types

We register three execution types corresponding to the pipeline stages: **StatisticsGen**, **SchemaGen**, and **AnomalyDetection**.

In [ ]:
# --- StatisticsGen execution type ---
stats_gen_exec_type = metadata_store_pb2.ExecutionType()
stats_gen_exec_type.name = 'StatisticsGen'
stats_gen_exec_type.properties['state'] = metadata_store_pb2.STRING
stats_gen_exec_type_id = store.put_execution_type(stats_gen_exec_type)

# --- SchemaGen execution type ---
schema_gen_exec_type = metadata_store_pb2.ExecutionType()
schema_gen_exec_type.name = 'SchemaGen'
schema_gen_exec_type.properties['state'] = metadata_store_pb2.STRING
schema_gen_exec_type_id = store.put_execution_type(schema_gen_exec_type)

# --- AnomalyDetection execution type ---
anomaly_exec_type = metadata_store_pb2.ExecutionType()
anomaly_exec_type.name = 'AnomalyDetection'
anomaly_exec_type.properties['state'] = metadata_store_pb2.STRING
anomaly_exec_type_id = store.put_execution_type(anomaly_exec_type)

print('Registered Execution Types:')
for et in store.get_execution_types():
    print(f'  - {et.name} (ID: {et.id})')

## Step 4: StatisticsGen — Generate & Record Training Statistics

### 4a. Create the input dataset artifact

In [ ]:
train_data_path = './data/train/data.csv'

data_artifact = metadata_store_pb2.Artifact()
data_artifact.uri = train_data_path
data_artifact.type_id = data_artifact_type_id
data_artifact.properties['name'].string_value = 'Wine Quality (Red) - Train'
data_artifact.properties['split'].string_value = 'train'
data_artifact.properties['version'].int_value = 1
data_artifact.properties['num_rows'].int_value = len(train_df)

data_artifact_id = store.put_artifacts([data_artifact])[0]
print(f'Train DataSet artifact ID: {data_artifact_id}')
print(data_artifact)

### 4b. Create the eval dataset artifact

In [ ]:
eval_data_path = './data/eval/data.csv'

eval_data_artifact = metadata_store_pb2.Artifact()
eval_data_artifact.uri = eval_data_path
eval_data_artifact.type_id = data_artifact_type_id
eval_data_artifact.properties['name'].string_value = 'Wine Quality (Red) - Eval'
eval_data_artifact.properties['split'].string_value = 'eval'
eval_data_artifact.properties['version'].int_value = 1
eval_data_artifact.properties['num_rows'].int_value = len(eval_df)

eval_data_artifact_id = store.put_artifacts([eval_data_artifact])[0]
print(f'Eval DataSet artifact ID: {eval_data_artifact_id}')
print(eval_data_artifact)

### 4c. Create execution, register input event, run TFDV, register output

In [ ]:
# Create StatisticsGen execution
stats_gen_exec = metadata_store_pb2.Execution()
stats_gen_exec.type_id = stats_gen_exec_type_id
stats_gen_exec.properties['state'].string_value = 'RUNNING'
stats_gen_exec_id = store.put_executions([stats_gen_exec])[0]

# Register input event (train dataset → StatisticsGen)
input_event = metadata_store_pb2.Event()
input_event.artifact_id = data_artifact_id
input_event.execution_id = stats_gen_exec_id
input_event.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([input_event])

print(f'StatisticsGen execution ID: {stats_gen_exec_id} (RUNNING)')
print(f'Input event: DataSet artifact {data_artifact_id} → Execution {stats_gen_exec_id}')

In [ ]:
# Run TFDV to generate training statistics
train_stats = tfdv.generate_statistics_from_csv(data_location=train_data_path)
tfdv.visualize_statistics(train_stats)

In [ ]:
# Create the output statistics artifact
stats_artifact = metadata_store_pb2.Artifact()
stats_artifact.uri = './stats/train_stats'
stats_artifact.type_id = stats_artifact_type_id
stats_artifact.properties['name'].string_value = 'Wine Quality Train Statistics'
stats_artifact.properties['split'].string_value = 'train'
stats_artifact.properties['version'].int_value = 1
stats_artifact_id = store.put_artifacts([stats_artifact])[0]

# Register output event (StatisticsGen → Statistics)
output_event = metadata_store_pb2.Event()
output_event.artifact_id = stats_artifact_id
output_event.execution_id = stats_gen_exec_id
output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([output_event])

# Mark execution as completed
stats_gen_exec.id = stats_gen_exec_id
stats_gen_exec.properties['state'].string_value = 'COMPLETED'
store.put_executions([stats_gen_exec])

print(f'Statistics artifact ID: {stats_artifact_id}')
print(f'StatisticsGen execution marked COMPLETED')

## Step 5: SchemaGen — Infer & Record Schema

We now use the generated statistics to infer a schema with TFDV, and track this as a separate execution in MLMD.

In [ ]:
# Create SchemaGen execution
schema_gen_exec = metadata_store_pb2.Execution()
schema_gen_exec.type_id = schema_gen_exec_type_id
schema_gen_exec.properties['state'].string_value = 'RUNNING'
schema_gen_exec_id = store.put_executions([schema_gen_exec])[0]

# Input event: Statistics → SchemaGen
schema_input_event = metadata_store_pb2.Event()
schema_input_event.artifact_id = stats_artifact_id
schema_input_event.execution_id = schema_gen_exec_id
schema_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT
store.put_events([schema_input_event])

print(f'SchemaGen execution ID: {schema_gen_exec_id} (RUNNING)')

In [ ]:
# Infer schema from statistics
schema = tfdv.infer_schema(statistics=train_stats)

schema_file = './schema.pbtxt'
tfdv.write_schema_text(schema, schema_file)

print(f"Schema written to: {schema_file}")
tfdv.display_schema(schema)

In [ ]:
# Create schema output artifact
schema_artifact = metadata_store_pb2.Artifact()
schema_artifact.uri = schema_file
schema_artifact.type_id = schema_artifact_type_id
schema_artifact.properties['name'].string_value = 'Wine Quality Schema'
schema_artifact.properties['version'].int_value = 1
schema_artifact_id = store.put_artifacts([schema_artifact])[0]

# Output event: SchemaGen → Schema
schema_output_event = metadata_store_pb2.Event()
schema_output_event.artifact_id = schema_artifact_id
schema_output_event.execution_id = schema_gen_exec_id
schema_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([schema_output_event])

# Mark execution as completed
schema_gen_exec.id = schema_gen_exec_id
schema_gen_exec.properties['state'].string_value = 'COMPLETED'
store.put_executions([schema_gen_exec])

print(f'Schema artifact ID: {schema_artifact_id}')
print('SchemaGen execution marked COMPLETED')

## Step 6: AnomalyDetection — Validate Eval Data Against Schema

This is an **additional step** not present in the original lab. We generate statistics for the eval split, compare it against the inferred schema, and record any anomalies.

In [ ]:
# Create AnomalyDetection execution
anomaly_exec = metadata_store_pb2.Execution()
anomaly_exec.type_id = anomaly_exec_type_id
anomaly_exec.properties['state'].string_value = 'RUNNING'
anomaly_exec_id = store.put_executions([anomaly_exec])[0]

# Input events: eval dataset + schema → AnomalyDetection
eval_input_event = metadata_store_pb2.Event()
eval_input_event.artifact_id = eval_data_artifact_id
eval_input_event.execution_id = anomaly_exec_id
eval_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

schema_input_event2 = metadata_store_pb2.Event()
schema_input_event2.artifact_id = schema_artifact_id
schema_input_event2.execution_id = anomaly_exec_id
schema_input_event2.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([eval_input_event, schema_input_event2])

print(f'AnomalyDetection execution ID: {anomaly_exec_id} (RUNNING)')

In [ ]:
# Generate eval statistics and validate against schema
eval_stats = tfdv.generate_statistics_from_csv(data_location=eval_data_path)
anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)

tfdv.display_anomalies(anomalies)

In [ ]:
# Determine if anomalies were found
has_anomalies = str(len(anomalies.anomaly_info) > 0)

# Create anomalies output artifact
anomalies_artifact = metadata_store_pb2.Artifact()
anomalies_artifact.uri = './anomalies/eval_anomalies'
anomalies_artifact.type_id = anomalies_artifact_type_id
anomalies_artifact.properties['name'].string_value = 'Eval Anomalies Report'
anomalies_artifact.properties['split'].string_value = 'eval'
anomalies_artifact.properties['has_anomalies'].string_value = has_anomalies
anomalies_artifact_id = store.put_artifacts([anomalies_artifact])[0]

# Output event: AnomalyDetection → Anomalies
anomaly_output_event = metadata_store_pb2.Event()
anomaly_output_event.artifact_id = anomalies_artifact_id
anomaly_output_event.execution_id = anomaly_exec_id
anomaly_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT
store.put_events([anomaly_output_event])

# Mark execution as completed
anomaly_exec.id = anomaly_exec_id
anomaly_exec.properties['state'].string_value = 'COMPLETED'
store.put_executions([anomaly_exec])

print(f'Anomalies artifact ID: {anomalies_artifact_id}')
print(f'Has anomalies: {has_anomalies}')
print('AnomalyDetection execution marked COMPLETED')

## Step 7: Visualize Train vs. Eval Statistics

Side-by-side comparison of train and eval statistics to understand distribution differences.

In [ ]:
tfdv.visualize_statistics(lhs_statistics=eval_stats, rhs_statistics=train_stats,
                          lhs_name='EVAL', rhs_name='TRAIN')

## Step 8: Set Up Context, Attributions, and Associations

We group all artifacts and executions under an **Experiment** context.

In [ ]:
# Create ContextType
expt_context_type = metadata_store_pb2.ContextType()
expt_context_type.name = 'Experiment'
expt_context_type.properties['note'] = metadata_store_pb2.STRING
expt_context_type.properties['pipeline_version'] = metadata_store_pb2.STRING
expt_context_type.properties['dataset_source'] = metadata_store_pb2.STRING
expt_context_type_id = store.put_context_type(expt_context_type)

# Create Context instance
expt_context = metadata_store_pb2.Context()
expt_context.type_id = expt_context_type_id
expt_context.name = 'Wine-Quality-MLMD-Experiment'
expt_context.properties['note'].string_value = (
    'MLMD walkthrough with Wine Quality dataset, '
    'tracking statistics generation, schema inference, and anomaly detection'
)
expt_context.properties['pipeline_version'].string_value = '1.0.0'
expt_context.properties['dataset_source'].string_value = (
    'https://archive.ics.uci.edu/ml/datasets/wine+quality'
)
expt_context_id = store.put_contexts([expt_context])[0]

print(f'Experiment Context ID: {expt_context_id}')
print(expt_context)

In [ ]:
# Create attributions (artifact ↔ context)
attributions = []
for aid in [data_artifact_id, eval_data_artifact_id,
            stats_artifact_id, schema_artifact_id, anomalies_artifact_id]:
    attr = metadata_store_pb2.Attribution()
    attr.artifact_id = aid
    attr.context_id = expt_context_id
    attributions.append(attr)

# Create associations (execution ↔ context)
associations = []
for eid in [stats_gen_exec_id, schema_gen_exec_id, anomaly_exec_id]:
    assoc = metadata_store_pb2.Association()
    assoc.execution_id = eid
    assoc.context_id = expt_context_id
    associations.append(assoc)

store.put_attributions_and_associations(attributions, associations)

print(f'Registered {len(attributions)} attributions and {len(associations)} associations')

## Step 9: Query the Metadata Store — Lineage Tracing

Now we demonstrate how to retrieve information from the metadata store to trace lineage. We'll answer the question: **"Which dataset was used to produce the schema?"**

In [ ]:
# List all artifact types in the store
print('=== Artifact Types ===')
for at in store.get_artifact_types():
    print(f'  {at.name} (ID: {at.id})')

print('\n=== Execution Types ===')
for et in store.get_execution_types():
    print(f'  {et.name} (ID: {et.id})')

print('\n=== All Artifacts ===')
for a in store.get_artifacts():
    print(f'  ID={a.id}, type_id={a.type_id}, uri={a.uri}')

In [ ]:
# Trace lineage: Schema → what produced it → what was the input?
schema_to_inv = store.get_artifacts_by_type('Schema')[0]
print(f'Investigating Schema artifact (ID: {schema_to_inv.id}):')
print(f'  Name: {schema_to_inv.properties["name"].string_value}')
print(f'  URI:  {schema_to_inv.uri}')

In [ ]:
# Find events related to the schema artifact
schema_events = store.get_events_by_artifact_ids([schema_to_inv.id])
print('Events related to the Schema artifact:')
for e in schema_events:
    event_type = metadata_store_pb2.Event.Type.Name(e.type)
    print(f'  Execution ID: {e.execution_id}, Type: {event_type}')

In [ ]:
# The schema is a DECLARED_OUTPUT of SchemaGen. Find the input to that execution.
schema_gen_execution_id = schema_events[0].execution_id
exec_events = store.get_events_by_execution_ids([schema_gen_execution_id])

print(f'Events for SchemaGen execution (ID: {schema_gen_execution_id}):')
for e in exec_events:
    event_type = metadata_store_pb2.Event.Type.Name(e.type)
    print(f'  Artifact ID: {e.artifact_id}, Type: {event_type}')

In [ ]:
# The DECLARED_INPUT to SchemaGen is the Statistics artifact.
# Now trace further: which dataset produced those statistics?
input_artifacts = [e for e in exec_events
                   if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
stats_art = store.get_artifacts_by_id([input_artifacts[0].artifact_id])[0]

print(f'SchemaGen input artifact (ID: {stats_art.id}):')
print(f'  Type: Statistics')
print(f'  Name: {stats_art.properties["name"].string_value}')

# Now find what produced the Statistics artifact
stats_events = store.get_events_by_artifact_ids([stats_art.id])
stats_gen_exec_events = store.get_events_by_execution_ids(
    [e.execution_id for e in stats_events if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT]
)

dataset_inputs = [e for e in stats_gen_exec_events
                  if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
source_dataset = store.get_artifacts_by_id([dataset_inputs[0].artifact_id])[0]

print(f'\nOriginal source dataset (ID: {source_dataset.id}):')
print(f'  Name: {source_dataset.properties["name"].string_value}')
print(f'  Split: {source_dataset.properties["split"].string_value}')
print(f'  URI: {source_dataset.uri}')
print(f'  Rows: {source_dataset.properties["num_rows"].int_value}')

### Query by Context

We can also retrieve all artifacts and executions associated with our experiment context.

In [ ]:
# Retrieve context
contexts = store.get_contexts_by_type('Experiment')
expt = contexts[0]
print(f'Experiment: {expt.name}')
print(f'  Note: {expt.properties["note"].string_value}')
print(f'  Pipeline Version: {expt.properties["pipeline_version"].string_value}')
print(f'  Dataset Source: {expt.properties["dataset_source"].string_value}')

# Get all artifacts in context
ctx_artifacts = store.get_artifacts_by_context(expt.id)
print(f'\nArtifacts in context ({len(ctx_artifacts)}):')
for a in ctx_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0].name
    print(f'  [{a_type}] ID={a.id}, URI={a.uri}')

# Get all executions in context
ctx_executions = store.get_executions_by_context(expt.id)
print(f'\nExecutions in context ({len(ctx_executions)}):')
for ex in ctx_executions:
    ex_type = store.get_execution_types_by_id([ex.type_id])[0].name
    print(f'  [{ex_type}] ID={ex.id}, State={ex.properties["state"].string_value}')

## Wrap Up

In this notebook, we demonstrated ML Metadata usage with several enhancements over the original lab:

1. **Wine Quality dataset** — a real-world UCI dataset with 12 physicochemical features instead of the Chicago Taxi dataset
2. **SQLite persistent store** — metadata survives across sessions, unlike the original fake database
3. **Three separate execution types** — StatisticsGen, SchemaGen, and AnomalyDetection — each properly tracked with input/output events
4. **Anomaly detection** — validates eval data against the inferred schema and records results
5. **Full lineage tracing** — demonstrated how to walk the event graph backwards from schema → statistics → dataset
6. **Context-based queries** — retrieved all artifacts and executions by experiment context

These techniques form the foundation for tracking ML pipelines in production, whether using TFX or custom systems.